# 04 Coding System and Conceptual Framework

This notebook builds Subagent 04's coding framework and Fig. 3 conceptual framework diagram. Inputs are limited to the new master data `data/NSFC正式增量采集_2014-2026_去重筛选最终结果.csv` and the new keyword table `data/NSFC正式增量采集_37个关键词.csv`; the coding denominator is the full master table, `N=9222`. Output logic is retained for later Fig. 3/table/log worker reruns:

- `output/tables/04_coding_framework.csv`
- `output/figures/Fig3_conceptual_framework.svg/pdf/tiff/png`
- `output/logs/04_coding_methods.md`

Note: all figure text is English as required; notebook comments and method notes are translated to English while data-semantic Chinese fields are retained where required.


In [ ]:
# Import the basic libraries required by this notebook.
# pandas reads the master data and keyword table and organizes the coding-framework table; matplotlib draws the vertical conceptual framework diagram;
# pathlib makes path construction more robust on macOS/Linux; datetime records method-log generation time.
from pathlib import Path
from datetime import datetime
from collections import Counter
import re

import numpy as np
import pandas as pd
import matplotlib as mpl
from matplotlib import font_manager
import matplotlib.pyplot as plt

# Register and require Times New Roman; stop immediately instead of using a fallback font if it is unavailable.
TIMES_NEW_ROMAN_PATHS = [
    Path("/Library/Fonts/Times New Roman.ttf"),
    Path("/Library/Fonts/Times New Roman Bold.ttf"),
    Path("/Library/Fonts/Times New Roman Italic.ttf"),
    Path("/Library/Fonts/Times New Roman Bold Italic.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Bold.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Italic.ttf"),
    Path("/System/Library/Fonts/Supplemental/Times New Roman Bold Italic.ttf"),
]
for font_path in TIMES_NEW_ROMAN_PATHS:
    if font_path.exists():
        font_manager.fontManager.addfont(str(font_path))
try:
    font_manager.findfont("Times New Roman", fallback_to_default=False)
except ValueError as exc:
    raise RuntimeError("Times New Roman is required for figure export but was not found by matplotlib.") from exc

from PIL import Image

# Set Matplotlib parameters commonly used for publication figures.
# svg.fonttype='none' preserves editable text in SVG, and pdf.fonttype=42 embeds TrueType fonts.
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.sans-serif": ["Times New Roman"],
    "font.monospace": ["Times New Roman"],
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "font.size": 8,
    "axes.spines.right": False,
    "axes.spines.top": False,
    "axes.linewidth": 0.8,
    "legend.frameon": False,
})

MAIN_DATA_FILENAME = 'NSFC正式增量采集_2014-2026_去重筛选最终结果.csv'
KEYWORD_TABLE_FILENAME = 'NSFC正式增量采集_37个关键词.csv'
EXPECTED_N_RECORDS = 9222
EXPECTED_KEYWORD_COUNT = 37

# Allow the notebook to run from scratch from either the project root or the code/ subdirectory.
# Prefer the directory that contains both the new master data and the new keyword table as the project root.
candidate_roots = [Path.cwd(), Path.cwd().parent, Path('/Volumes/ZHITAI2T/202607综述BAE')]
PROJECT_ROOT = None
for candidate in candidate_roots:
    data_dir = candidate / 'data'
    if (data_dir / MAIN_DATA_FILENAME).exists() and (data_dir / KEYWORD_TABLE_FILENAME).exists():
        PROJECT_ROOT = candidate.resolve()
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f'Could not locate data/{MAIN_DATA_FILENAME} and data/{KEYWORD_TABLE_FILENAME}; '
        'run this notebook from the project root or the code/ directory.'
    )

DATA_PATH = PROJECT_ROOT / 'data' / MAIN_DATA_FILENAME
KEYWORD_PATH = PROJECT_ROOT / 'data' / KEYWORD_TABLE_FILENAME
FIG_DIR = PROJECT_ROOT / 'output' / 'figures'
TABLE_DIR = PROJECT_ROOT / 'output' / 'tables'
LOG_DIR = PROJECT_ROOT / 'output' / 'logs'
NOTEBOOK_PATH = PROJECT_ROOT / 'code' / '04_coding_system_and_conceptual_framework.ipynb'

# Create only the output directories owned by this subagent; do not modify data/ or files owned by other subagents.
for path in [FIG_DIR, TABLE_DIR, LOG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

FIG_BASENAME = FIG_DIR / 'Fig3_conceptual_framework'
FRAMEWORK_CSV = TABLE_DIR / '04_coding_framework.csv'
METHOD_LOG = LOG_DIR / '04_coding_methods.md'

print(f'Project root: {PROJECT_ROOT}')
print(f'Input master data: {DATA_PATH}')
print(f'Input keyword table: {KEYWORD_PATH}')


In [ ]:
# Read the new keyword table and new master data, then check whether fields required by this task are present.
keyword_df = pd.read_csv(KEYWORD_PATH)
keyword_df.columns = [str(col).strip() for col in keyword_df.columns]

REQUIRED_KEYWORD_COLUMNS = ['term', 'term_group']
missing_keyword_columns = [col for col in REQUIRED_KEYWORD_COLUMNS if col not in keyword_df.columns]
if missing_keyword_columns:
    raise ValueError(f'Keyword table is missing required columns: {missing_keyword_columns}')

keyword_df = keyword_df[REQUIRED_KEYWORD_COLUMNS].copy()
keyword_df['term'] = keyword_df['term'].astype(str).str.strip()
keyword_df['term_group'] = keyword_df['term_group'].astype(str).str.strip()
keyword_df = keyword_df[keyword_df['term'] != '']

if len(keyword_df) != EXPECTED_KEYWORD_COUNT:
    raise ValueError(f'Keyword table should contain {EXPECTED_KEYWORD_COUNT} keywords; current count is {len(keyword_df)}.')
if keyword_df['term'].duplicated().any():
    duplicated_terms = sorted(keyword_df.loc[keyword_df['term'].duplicated(), 'term'].unique())
    raise ValueError(f'Keyword table contains duplicated term values: {duplicated_terms}')

EXPECTED_KEYWORD_GROUPS = {'method', 'object_built_environment', 'performance_environment'}
actual_keyword_groups = set(keyword_df['term_group'])
if actual_keyword_groups != EXPECTED_KEYWORD_GROUPS:
    raise ValueError(f'Keyword table term_group should be {EXPECTED_KEYWORD_GROUPS}; current groups are {actual_keyword_groups}')

keyword_terms = set(keyword_df['term'])
keyword_terms_by_group = {
    group: set(group_df['term'])
    for group, group_df in keyword_df.groupby('term_group')
}

df = pd.read_csv(DATA_PATH)

REQUIRED_COLUMNS = [
    'matched_method_terms',
    'matched_object_terms',
    'matched_performance_terms',
]
missing_columns = [col for col in REQUIRED_COLUMNS if col not in df.columns]
if missing_columns:
    raise ValueError(f'Master data is missing required columns: {missing_columns}')

n_records = len(df)
if n_records != EXPECTED_N_RECORDS:
    raise ValueError(f'Coding denominator should be N={EXPECTED_N_RECORDS}; current master-table record count is {n_records}.')

year_min = int(df['award_year'].min()) if 'award_year' in df.columns and df['award_year'].notna().any() else None
year_max = int(df['award_year'].max()) if 'award_year' in df.columns and df['award_year'].notna().any() else None

print(f'Master-table records N={n_records}')
print(f'Year range: {year_min}-{year_max}')
print(f'Keyword table: {len(keyword_df)} keywords; groups={keyword_df["term_group"].value_counts().to_dict()}')
print(df[REQUIRED_COLUMNS].head())


In [ ]:
# Parse semicolon-separated matched terms into sets.
# The data may contain either Chinese or English semicolons; normalize to English semicolons before splitting.
def split_terms(value):
    """Split matched terms in a cell into a stripped set with nan-like values removed."""
    if pd.isna(value):
        return set()
    terms = []
    for term in str(value).replace('；', ';').split(';'):
        term = term.strip()
        if term and term.lower() != 'nan':
            terms.append(term)
    return set(terms)

# Attach the three term-set columns to each record; later coding rules are derived from these columns.
term_df = df.copy()
term_df['method_term_set'] = term_df['matched_method_terms'].apply(split_terms)
term_df['object_term_set'] = term_df['matched_object_terms'].apply(split_terms)
term_df['performance_term_set'] = term_df['matched_performance_terms'].apply(split_terms)

# Count each raw term frequency to verify that the coding framework covers the keywords actually observed in the data.
def count_terms(series):
    counter = Counter()
    for terms in series:
        counter.update(terms)
    return counter

method_counts = count_terms(term_df['method_term_set'])
object_counts = count_terms(term_df['object_term_set'])
performance_counts = count_terms(term_df['performance_term_set'])

observed_terms_by_group = {
    'method': set(method_counts),
    'object_built_environment': set(object_counts),
    'performance_environment': set(performance_counts),
}
unexpected_terms_by_group = {
    group: sorted(observed_terms - keyword_terms_by_group[group])
    for group, observed_terms in observed_terms_by_group.items()
}
unexpected_terms_by_group = {group: terms for group, terms in unexpected_terms_by_group.items() if terms}
if unexpected_terms_by_group:
    raise ValueError(f'Master-table matched terms contain terms outside the corresponding groups in the new keyword table: {unexpected_terms_by_group}')

all_observed_terms = set().union(*observed_terms_by_group.values())
unobserved_keyword_terms = sorted(keyword_terms - all_observed_terms)
if unobserved_keyword_terms:
    raise ValueError(f'Keyword table contains terms not observed in master-table matched terms: {unobserved_keyword_terms}')

print('Method/data term frequencies:', method_counts.most_common())
print('Object term frequencies:', object_counts.most_common())
print('Performance term frequencies:', performance_counts.most_common())
print('Master-table matched terms are consistent with the new 37-keyword table.')


In [ ]:
# Five-dimensional coding framework definition.
# The first four dimensions are non-exclusive: one record can hit multiple categories within a dimension, supporting later co-occurrence networks, cross-tabs, and heatmaps.
# The fifth dimension, "Application maturity", derives the primary stage: each record is assigned to one maturity stage by priority, supporting stage distributions and review narrative structure.

framework_specs = [
    # 1. Built-environment object
    {
        'dimension_id': 'D1',
        'dimension_en': 'Built-environment object',
        'dimension_cn': '建成环境对象',
        'category_id': 'D1-C1',
        'category_en': 'Regional urban systems',
        'category_cn': '区域/城市群系统',
        'source_column': 'matched_object_terms',
        'keywords': ['城市群'],
        'coding_rule_cn': 'matched_object_terms 命中“城市群”。',
        'analysis_use_cn': '用于识别区域尺度、都市圈与城市群层面的建成环境绩效研究。',
        'coding_mode': 'non-exclusive keyword hit',
    },
    {
        'dimension_id': 'D1',
        'dimension_en': 'Built-environment object',
        'dimension_cn': '建成环境对象',
        'category_id': 'D1-C2',
        'category_en': 'Planning and spatial governance',
        'category_cn': '规划与空间治理对象',
        'source_column': 'matched_object_terms',
        'keywords': ['规划'],
        'coding_rule_cn': 'matched_object_terms 命中“规划”。',
        'analysis_use_cn': '用于区分规划管控、政策应用和空间治理导向研究。',
        'coding_mode': 'non-exclusive keyword hit',
    },
    {
        'dimension_id': 'D1',
        'dimension_en': 'Built-environment object',
        'dimension_cn': '建成环境对象',
        'category_id': 'D1-C3',
        'category_en': 'Land use and urban form',
        'category_cn': '土地利用与城市形态',
        'source_column': 'matched_object_terms',
        'keywords': ['土地利用', '城市形态', '建成环境'],
        'coding_rule_cn': 'matched_object_terms 命中“土地利用”“城市形态”或“建成环境”。',
        'analysis_use_cn': '用于分析形态、功能布局与土地利用变化如何连接环境绩效。',
        'coding_mode': 'non-exclusive keyword hit',
    },
    {
        'dimension_id': 'D1',
        'dimension_en': 'Built-environment object',
        'dimension_cn': '建成环境对象',
        'category_id': 'D1-C4',
        'category_en': 'Buildings and vertical urban fabric',
        'category_cn': '建筑与垂直城市肌理',
        'source_column': 'matched_object_terms',
        'keywords': ['建筑'],
        'coding_rule_cn': 'matched_object_terms 命中“建筑”。',
        'analysis_use_cn': '用于定位建筑尺度、三维形态、城市通风和能耗相关研究。',
        'coding_mode': 'non-exclusive keyword hit',
    },
    {
        'dimension_id': 'D1',
        'dimension_en': 'Built-environment object',
        'dimension_cn': '建成环境对象',
        'category_id': 'D1-C5',
        'category_en': 'Streets and neighbourhood blocks',
        'category_cn': '街道与街区',
        'source_column': 'matched_object_terms',
        'keywords': ['街道', '街区'],
        'coding_rule_cn': 'matched_object_terms 命中“街道”或“街区”。',
        'analysis_use_cn': '用于识别街道空间、街区组织、活动行为与暴露测度研究。',
        'coding_mode': 'non-exclusive keyword hit',
    },
    {
        'dimension_id': 'D1',
        'dimension_en': 'Built-environment object',
        'dimension_cn': '建成环境对象',
        'category_id': 'D1-C6',
        'category_en': 'Green-blue and sponge infrastructure',
        'category_cn': '绿蓝与海绵基础设施',
        'source_column': 'matched_object_terms',
        'keywords': ['绿地', '海绵'],
        'coding_rule_cn': 'matched_object_terms 命中“绿地”或“海绵”。',
        'analysis_use_cn': '用于组织绿地降温、生态服务、雨洪韧性和海绵城市相关证据。',
        'coding_mode': 'non-exclusive keyword hit',
    },

    # 2. Sensing data source
    {
        'dimension_id': 'D2',
        'dimension_en': 'Sensing data source',
        'dimension_cn': '感知数据源',
        'category_id': 'D2-C1',
        'category_en': 'Satellite and aerial imagery',
        'category_cn': '卫星/航空遥感影像',
        'source_column': 'matched_method_terms',
        'keywords': ['遥感', '高分'],
        'coding_rule_cn': 'matched_method_terms 命中“遥感”或“高分”。',
        'analysis_use_cn': '用于区分以影像反演、土地覆盖识别和热环境遥感为基础的研究。',
        'coding_mode': 'non-exclusive keyword hit',
    },
    {
        'dimension_id': 'D2',
        'dimension_en': 'Sensing data source',
        'dimension_cn': '感知数据源',
        'category_id': 'D2-C2',
        'category_en': 'Night-time light products',
        'category_cn': '夜间灯光产品',
        'source_column': 'matched_method_terms',
        'keywords': ['夜间灯光'],
        'coding_rule_cn': 'matched_method_terms 命中“夜间灯光”。',
        'analysis_use_cn': '用于识别城市活动强度、能源/碳排放代理变量和空间扩张测度。',
        'coding_mode': 'non-exclusive keyword hit',
    },
    {
        'dimension_id': 'D2',
        'dimension_en': 'Sensing data source',
        'dimension_cn': '感知数据源',
        'category_id': 'D2-C3',
        'category_en': 'Geospatial and GIS databases',
        'category_cn': '地理空间与 GIS 数据',
        'source_column': 'matched_method_terms',
        'keywords': ['GIS', '地理信息'],
        'coding_rule_cn': 'matched_method_terms 命中“GIS”或“地理信息”。',
        'analysis_use_cn': '用于标记空间叠加、地理建模、空间统计和基础地理数据驱动研究。',
        'coding_mode': 'non-exclusive keyword hit',
    },
    {
        'dimension_id': 'D2',
        'dimension_en': 'Sensing data source',
        'dimension_cn': '感知数据源',
        'category_id': 'D2-C4',
        'category_en': 'Multi-source urban big data',
        'category_cn': '多源城市大数据',
        'source_column': 'matched_method_terms',
        'keywords': ['多源数据', '大数据'],
        'coding_rule_cn': 'matched_method_terms 命中“多源数据”或“大数据”。',
        'analysis_use_cn': '用于识别多源融合、平台化城市数据和综合环境绩效测度研究。',
        'coding_mode': 'non-exclusive keyword hit',
    },
    {
        'dimension_id': 'D2',
        'dimension_en': 'Sensing data source',
        'dimension_cn': '感知数据源',
        'category_id': 'D2-C5',
        'category_en': 'Street-level and POI data',
        'category_cn': '街景与 POI 数据',
        'source_column': 'matched_method_terms',
        'keywords': ['街景', 'POI'],
        'coding_rule_cn': 'matched_method_terms 命中“街景”或“POI”。',
        'analysis_use_cn': '用于支持微观街道品质、功能混合、可达性和人本暴露研究。',
        'coding_mode': 'non-exclusive keyword hit',
    },
    {
        'dimension_id': 'D2',
        'dimension_en': 'Sensing data source',
        'dimension_cn': '感知数据源',
        'category_id': 'D2-C6',
        'category_en': 'Mobility and location traces',
        'category_cn': '移动性与位置轨迹',
        'source_column': 'matched_method_terms',
        'keywords': ['手机信令', '轨迹', 'LBS'],
        'coding_rule_cn': 'matched_method_terms 命中“手机信令”“轨迹”或“LBS”。',
        'analysis_use_cn': '用于连接居民行为、交通活动、时空暴露和低碳出行应用。',
        'coding_mode': 'non-exclusive keyword hit',
    },
    {
        'dimension_id': 'D2',
        'dimension_en': 'Sensing data source',
        'dimension_cn': '感知数据源',
        'category_id': 'D2-C7',
        'category_en': '3D and LiDAR data',
        'category_cn': '三维与 LiDAR 数据',
        'source_column': 'matched_method_terms',
        'keywords': ['三维', 'LiDAR'],
        'coding_rule_cn': 'matched_method_terms 命中“三维”或“LiDAR”。',
        'analysis_use_cn': '用于识别城市三维形态、建筑体量、通风和微气候模拟相关研究。',
        'coding_mode': 'non-exclusive keyword hit',
    },

    # 3. Analytical method
    {
        'dimension_id': 'D3',
        'dimension_en': 'Analytical method',
        'dimension_cn': '分析方法',
        'category_id': 'D3-C1',
        'category_en': 'Machine learning, deep learning and AI',
        'category_cn': '机器学习/深度学习/人工智能',
        'source_column': 'matched_method_terms',
        'keywords': ['机器学习', '深度学习', '人工智能'],
        'coding_rule_cn': 'matched_method_terms 命中“机器学习”“深度学习”或“人工智能”。',
        'analysis_use_cn': '用于区分预测、分类、识别和非线性机制学习类研究。',
        'coding_mode': 'non-exclusive keyword hit',
    },
    {
        'dimension_id': 'D3',
        'dimension_en': 'Analytical method',
        'dimension_cn': '分析方法',
        'category_id': 'D3-C2',
        'category_en': 'Spatial analysis and GIS modelling',
        'category_cn': '空间分析与 GIS 建模',
        'source_column': 'matched_method_terms',
        'keywords': ['GIS', '地理信息'],
        'coding_rule_cn': 'matched_method_terms 命中“GIS”或“地理信息”。',
        'analysis_use_cn': '用于组织空间叠加、空间关联、可达性和区域格局分析。',
        'coding_mode': 'non-exclusive keyword hit',
    },
    {
        'dimension_id': 'D3',
        'dimension_en': 'Analytical method',
        'dimension_cn': '分析方法',
        'category_id': 'D3-C3',
        'category_en': 'Remote-sensing retrieval and interpretation',
        'category_cn': '遥感反演与影像解译',
        'source_column': 'matched_method_terms',
        'keywords': ['遥感', '高分', '夜间灯光'],
        'coding_rule_cn': 'matched_method_terms 命中“遥感”“高分”或“夜间灯光”。',
        'analysis_use_cn': '用于汇总影像解译、参数反演、土地覆盖/热环境识别方法。',
        'coding_mode': 'non-exclusive keyword hit',
    },
    {
        'dimension_id': 'D3',
        'dimension_en': 'Analytical method',
        'dimension_cn': '分析方法',
        'category_id': 'D3-C4',
        'category_en': 'Multi-source data fusion',
        'category_cn': '多源数据融合',
        'source_column': 'matched_method_terms',
        'keywords': ['多源数据', '大数据'],
        'coding_rule_cn': 'matched_method_terms 命中“多源数据”或“大数据”。',
        'analysis_use_cn': '用于比较多源感知、数据融合和综合指标构建研究。',
        'coding_mode': 'non-exclusive keyword hit',
    },
    {
        'dimension_id': 'D3',
        'dimension_en': 'Analytical method',
        'dimension_cn': '分析方法',
        'category_id': 'D3-C5',
        'category_en': '3D modelling and simulation',
        'category_cn': '三维建模与模拟',
        'source_column': 'matched_method_terms',
        'keywords': ['三维', 'LiDAR'],
        'coding_rule_cn': 'matched_method_terms 命中“三维”或“LiDAR”。',
        'analysis_use_cn': '用于识别三维城市形态、建筑体量、CFD/微气候和情景模拟研究。',
        'coding_mode': 'non-exclusive keyword hit',
    },
    {
        'dimension_id': 'D3',
        'dimension_en': 'Analytical method',
        'dimension_cn': '分析方法',
        'category_id': 'D3-C6',
        'category_en': 'Mobility and activity pattern mining',
        'category_cn': '移动行为与活动模式挖掘',
        'source_column': 'matched_method_terms',
        'keywords': ['轨迹', '手机信令', 'LBS', 'POI'],
        'coding_rule_cn': 'matched_method_terms 命中“轨迹”“手机信令”“LBS”或“POI”。',
        'analysis_use_cn': '用于分析出行、活动空间、时空行为和动态暴露测度。',
        'coding_mode': 'non-exclusive keyword hit',
    },

    # 4. Environmental performance
    {
        'dimension_id': 'D4',
        'dimension_en': 'Environmental performance',
        'dimension_cn': '环境绩效',
        'category_id': 'D4-C1',
        'category_en': 'Urban climate and heat',
        'category_cn': '城市气候与热环境',
        'source_column': 'matched_performance_terms',
        'keywords': ['气候', '热岛', '热环境'],
        'coding_rule_cn': 'matched_performance_terms 命中“气候”“热岛”或“热环境”。',
        'analysis_use_cn': '用于组织热岛、微气候、热舒适和气候适应相关研究。',
        'coding_mode': 'non-exclusive keyword hit',
    },
    {
        'dimension_id': 'D4',
        'dimension_en': 'Environmental performance',
        'dimension_cn': '环境绩效',
        'category_id': 'D4-C2',
        'category_en': 'Carbon and energy performance',
        'category_cn': '碳与能源绩效',
        'source_column': 'matched_performance_terms',
        'keywords': ['碳', '能源'],
        'coding_rule_cn': 'matched_performance_terms 命中“碳”或“能源”。',
        'analysis_use_cn': '用于归纳低碳城市、建筑能耗、能源系统和排放代理测度研究。',
        'coding_mode': 'non-exclusive keyword hit',
    },
    {
        'dimension_id': 'D4',
        'dimension_en': 'Environmental performance',
        'dimension_cn': '环境绩效',
        'category_id': 'D4-C3',
        'category_en': 'Air quality and exposure',
        'category_cn': '空气质量与暴露',
        'source_column': 'matched_performance_terms',
        'keywords': ['空气污染', '暴露'],
        'coding_rule_cn': 'matched_performance_terms 命中“空气污染”或“暴露”。',
        'analysis_use_cn': '用于连接污染浓度、健康暴露、街道环境和活动空间研究。',
        'coding_mode': 'non-exclusive keyword hit',
    },
    {
        'dimension_id': 'D4',
        'dimension_en': 'Environmental performance',
        'dimension_cn': '环境绩效',
        'category_id': 'D4-C4',
        'category_en': 'Ecological quality',
        'category_cn': '生态环境质量',
        'source_column': 'matched_performance_terms',
        'keywords': ['生态环境'],
        'coding_rule_cn': 'matched_performance_terms 命中“生态环境”。',
        'analysis_use_cn': '用于识别生态质量、生态服务和城市生态安全评价研究。',
        'coding_mode': 'non-exclusive keyword hit',
    },
    {
        'dimension_id': 'D4',
        'dimension_en': 'Environmental performance',
        'dimension_cn': '环境绩效',
        'category_id': 'D4-C5',
        'category_en': 'Flooding and resilience',
        'category_cn': '洪涝与韧性',
        'source_column': 'matched_performance_terms',
        'keywords': ['洪涝', '韧性'],
        'coding_rule_cn': 'matched_performance_terms 命中“洪涝”或“韧性”。',
        'analysis_use_cn': '用于总结雨洪风险、海绵城市、韧性评价和适应性规划研究。',
        'coding_mode': 'non-exclusive keyword hit',
    },

    # 5. Application maturity
    {
        'dimension_id': 'D5',
        'dimension_en': 'Application maturity',
        'dimension_cn': '应用成熟度',
        'category_id': 'D5-C1',
        'category_en': 'Exploratory sensing and data inventory',
        'category_cn': '探索性感知与数据清查',
        'source_column': 'derived from three matched-term columns',
        'keywords': ['matched_method_terms only'],
        'coding_rule_cn': '若仅有方法/数据词，缺少明确对象词和绩效词，则归为探索性感知与数据清查。',
        'analysis_use_cn': '用于识别仍处于数据获取、数据清查和主题探索阶段的研究。',
        'coding_mode': 'exclusive derived stage',
    },
    {
        'dimension_id': 'D5',
        'dimension_en': 'Application maturity',
        'dimension_cn': '应用成熟度',
        'category_id': 'D5-C2',
        'category_en': 'Built-environment mapping and characterization',
        'category_cn': '建成环境制图与特征刻画',
        'source_column': 'derived from three matched-term columns',
        'keywords': ['object terms without performance terms'],
        'coding_rule_cn': '若有建成环境对象词但无明确环境绩效词，且未进入更高优先级类别，则归为制图与特征刻画。',
        'analysis_use_cn': '用于识别从数据感知走向对象识别、形态刻画和空间结构描述的研究。',
        'coding_mode': 'exclusive derived stage',
    },
    {
        'dimension_id': 'D5',
        'dimension_en': 'Application maturity',
        'dimension_cn': '应用成熟度',
        'category_id': 'D5-C3',
        'category_en': 'Performance measurement and assessment',
        'category_cn': '绩效测度与评价',
        'source_column': 'derived from three matched-term columns',
        'keywords': ['object terms + performance terms'],
        'coding_rule_cn': '若对象词和绩效词同时出现，且未命中模型预测或规划应用的更高优先级规则，则归为绩效测度与评价。',
        'analysis_use_cn': '用于识别可直接进入对象-绩效交叉分析的测度型研究。',
        'coding_mode': 'exclusive derived stage',
    },
    {
        'dimension_id': 'D5',
        'dimension_en': 'Application maturity',
        'dimension_cn': '应用成熟度',
        'category_id': 'D5-C4',
        'category_en': 'Model-based prediction and mechanism inference',
        'category_cn': '模型预测与机制解释',
        'source_column': 'derived from three matched-term columns',
        'keywords': ['人工智能', '机器学习', '深度学习', '三维', 'LiDAR', '多源数据'],
        'coding_rule_cn': '若方法词含 AI/机器学习/深度学习/三维/LiDAR/多源数据，且存在对象词或绩效词，并且未命中规划应用优先规则，则归为模型预测与机制解释。',
        'analysis_use_cn': '用于识别从测度走向预测、模拟、机制解释和多源融合建模的研究。',
        'coding_mode': 'exclusive derived stage',
    },
    {
        'dimension_id': 'D5',
        'dimension_en': 'Application maturity',
        'dimension_cn': '应用成熟度',
        'category_id': 'D5-C5',
        'category_en': 'Decision support and planning application',
        'category_cn': '决策支持与规划应用',
        'source_column': 'derived from three matched-term columns',
        'keywords': ['规划 + performance terms', '海绵 + 洪涝/韧性'],
        'coding_rule_cn': '若对象词含“规划”且有绩效词，或“海绵”与“洪涝/韧性”同时出现，则优先归为决策支持与规划应用。',
        'analysis_use_cn': '用于识别最接近规划策略、情景优化、治理应用和政策转化的研究。',
        'coding_mode': 'exclusive derived stage',
    },
]

# Validate that the first four non-exclusive coding dimensions use only the 37 terms from the new keyword table.
framework_keyword_terms = {
    keyword
    for spec in framework_specs
    if spec['coding_mode'] == 'non-exclusive keyword hit'
    for keyword in spec['keywords']
}
unknown_framework_keywords = sorted(framework_keyword_terms - keyword_terms)
if unknown_framework_keywords:
    raise ValueError(f'Coding framework contains keywords not found in the new keyword table: {unknown_framework_keywords}')

uncovered_keyword_terms = sorted(keyword_terms - framework_keyword_terms)
if uncovered_keyword_terms:
    raise ValueError(f'Terms in the new keyword table are not covered by the first four coding-framework dimensions: {uncovered_keyword_terms}')

print(f'Coding-framework category count: {len(framework_specs)}')
print(f'Keyword coverage in the first four dimensions: {len(framework_keyword_terms)}/{len(keyword_terms)}')

In [ ]:
# Helper function: determine whether a record's term set hits any keyword in a list.
def hit_any(term_set, keywords):
    return bool(set(keywords) & set(term_set))

# Primary-stage derivation function for application maturity.
# This dimension is not an existing field in the raw matched terms; it is rule-derived from combinations of the three matched-term groups.
# Rules are prioritized from higher to lower maturity: planning application > model prediction/mechanism inference > performance assessment > object characterization > data inventory.
def assign_application_maturity(row):
    method_terms = row['method_term_set']
    object_terms = row['object_term_set']
    performance_terms = row['performance_term_set']

    ai_model_terms = {'人工智能', '机器学习', '深度学习'}
    simulation_terms = {'三维', 'LiDAR'}
    fusion_terms = {'多源数据'}
    flood_resilience_terms = {'洪涝', '韧性'}

    if ('规划' in object_terms and performance_terms) or ({'海绵'} & object_terms and flood_resilience_terms & performance_terms):
        return 'Decision support and planning application'
    if (ai_model_terms | simulation_terms | fusion_terms) & method_terms and (object_terms or performance_terms):
        return 'Model-based prediction and mechanism inference'
    if object_terms and performance_terms:
        return 'Performance measurement and assessment'
    if object_terms and not performance_terms:
        return 'Built-environment mapping and characterization'
    return 'Exploratory sensing and data inventory'

term_df['application_maturity'] = term_df.apply(assign_application_maturity, axis=1)
maturity_counts = term_df['application_maturity'].value_counts().to_dict()

# Calculate category-hit record counts for the first four non-exclusive dimensions; the fifth dimension uses exclusive primary-stage counts.
framework_rows = []
for spec in framework_specs:
    row = dict(spec)
    if spec['dimension_id'] == 'D5':
        n_hit = maturity_counts.get(spec['category_en'], 0)
        observed_terms = '; '.join(spec['keywords'])
    else:
        if spec['source_column'] == 'matched_method_terms':
            term_col = 'method_term_set'
        elif spec['source_column'] == 'matched_object_terms':
            term_col = 'object_term_set'
        elif spec['source_column'] == 'matched_performance_terms':
            term_col = 'performance_term_set'
        else:
            raise ValueError(f"Unknown term source column: {spec['source_column']}")
        mask = term_df[term_col].apply(lambda terms: hit_any(terms, spec['keywords']))
        n_hit = int(mask.sum())

        # List only keywords in this category that actually appear in the data, supporting later checks of rule coverage.
        observed = []
        for keyword in spec['keywords']:
            if term_df[term_col].apply(lambda terms: keyword in terms).any():
                observed.append(keyword)
        observed_terms = '; '.join(observed)

    row['keyword_rule'] = '; '.join(spec['keywords'])
    row['observed_terms'] = observed_terms
    row['n_records'] = n_hit
    row['pct_records'] = round(n_hit / n_records * 100, 1) if n_records else 0.0
    framework_rows.append(row)

framework_table = pd.DataFrame(framework_rows)

# Keep output column order stable so other subagents or later analysis scripts can read it reliably.
framework_table = framework_table[
    [
        'dimension_id', 'dimension_en', 'dimension_cn',
        'category_id', 'category_en', 'category_cn',
        'source_column', 'coding_mode', 'keyword_rule', 'observed_terms',
        'coding_rule_cn', 'analysis_use_cn', 'n_records', 'pct_records',
    ]
]

framework_table.to_csv(FRAMEWORK_CSV, index=False, encoding='utf-8-sig')
print(f'Saved coding-framework table: {FRAMEWORK_CSV}')
framework_table

In [ ]:
# Summarize the number of categories in each dimension for the method log and final reporting.
dimension_summary = (
    framework_table
    .groupby(['dimension_id', 'dimension_en', 'dimension_cn'], as_index=False)
    .agg(n_categories=('category_id', 'nunique'), total_category_hits=('n_records', 'sum'))
)

dimension_summary

In [ ]:
# Draw the Fig. 3 conceptual framework diagram.
# Style follows Fig. 2: black outlines, no filled background, and enlarged English labels.
# This figure is a schematic-led composite: it emphasizes the five-layer conceptual framework of the review rather than showing numeric statistics or topic-result tables.
from matplotlib.patches import Rectangle


def draw_conceptual_framework(output_base):
    """Draw and export the horizontally compressed conceptual framework diagram."""
    modules = [
        {
            'title': 'Sensing data sources',
            'rows': [
                ['Satellite imagery', 'Night-time lights', 'GIS data', 'Urban big data'],
                ['Street-view / POI', 'Mobility traces', '3D / LiDAR'],
            ],
            'fill': 'none',
        },
        {
            'title': 'Analytical methods',
            'rows': [
                ['ML / DL / AI', 'Spatial analysis', 'RS retrieval', 'Data fusion', '3D modelling', 'Pattern mining'],
            ],
            'fill': 'none',
        },
        {
            'title': 'Built environment objects',
            'rows': [
                ['Regional systems', 'Planning systems', 'Land use / form', 'Buildings', 'Streets / blocks', 'Green-blue infra.'],
            ],
            'fill': 'none',
        },
        {
            'title': 'Environmental\nperformance',
            'rows': [
                ['Climate / heat', 'Carbon / energy', 'Air / exposure', 'Ecological quality', 'Flood / resilience'],
            ],
            'fill': 'none',
        },
        {
            'title': 'Application maturity',
            'rows': [
                ['Inventory', 'Mapping', 'Measurement', 'Prediction', 'Decision support'],
            ],
            'fill': 'none',
        },
    ]

    fig, ax = plt.subplots(figsize=(9.7, 4.15), dpi=300)
    fig.patch.set_facecolor('none')
    ax.set_facecolor('none')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

    left, right = 0.035, 0.965
    bottom, top = 0.055, 0.945
    gap = 0.011
    box_w = right - left
    box_h = (top - bottom - gap * (len(modules) - 1)) / len(modules)
    sep_x = left + 0.255
    chip_left = sep_x + 0.022
    chip_right = right - 0.022
    chip_gap = 0.009
    chip_h = 0.038
    chip_row_gap = 0.016

    def draw_chip_row(items, y_center, facecolor):
        n_items = len(items)
        total_gap = chip_gap * (n_items - 1)
        chip_w = (chip_right - chip_left - total_gap) / n_items
        for item_index, item in enumerate(items):
            x0 = chip_left + item_index * (chip_w + chip_gap)
            chip = Rectangle(
                (x0, y_center - chip_h / 2), chip_w, chip_h,
                linewidth=0.55,
                edgecolor='#111111',
                facecolor=facecolor,
                joinstyle='miter',
            )
            ax.add_patch(chip)
            ax.text(
                x0 + chip_w / 2, y_center,
                item,
                ha='center', va='center',
                fontsize=7.5,
                color='#111111',
            )

    for idx, module in enumerate(modules, start=1):
        box_y = top - idx * box_h - (idx - 1) * gap
        rect = Rectangle(
            (left, box_y), box_w, box_h,
            linewidth=1.0,
            edgecolor='#111111',
            facecolor=module['fill'],
            joinstyle='miter',
        )
        ax.add_patch(rect)

        ax.plot(
            [sep_x, sep_x], [box_y, box_y + box_h],
            color='#111111', linewidth=0.75, solid_capstyle='butt'
        )

        header_y = box_y + box_h / 2
        ax.text(
            left + 0.021, header_y,
            f'{idx:02d}',
            ha='left', va='center',
            fontsize=8.75, fontweight='bold',
            color='#111111',
        )
        ax.text(
            left + 0.061, header_y,
            module['title'],
            ha='left', va='center',
            fontsize=8.75, fontweight='bold',
            color='#111111', linespacing=0.92,
        )

        rows = module['rows']
        n_rows = len(rows)
        total_chip_h = n_rows * chip_h + (n_rows - 1) * chip_row_gap
        start_y = box_y + box_h / 2 + total_chip_h / 2 - chip_h / 2
        chip_face = 'none'
        for row_index, row_items in enumerate(rows):
            y_center = start_y - row_index * (chip_h + chip_row_gap)
            draw_chip_row(row_items, y_center, chip_face)

    fig.savefig(f'{output_base}.svg', bbox_inches='tight', pad_inches=0.035, facecolor='none', transparent=True)
    fig.savefig(f'{output_base}.pdf', bbox_inches='tight', pad_inches=0.035, facecolor='none', transparent=True)
    fig.savefig(f'{output_base}.tiff', dpi=600, bbox_inches='tight', pad_inches=0.035, facecolor='white')
    fig.savefig(f'{output_base}.png', dpi=600, bbox_inches='tight', pad_inches=0.035, facecolor='white')
    return fig


fig = draw_conceptual_framework(FIG_BASENAME)
plt.show()

print('Exported Fig. 3:')
for suffix in ['svg', 'pdf', 'tiff', 'png']:
    out = Path(f'{FIG_BASENAME}.{suffix}')
    print(f' - {out} ({out.stat().st_size:,} bytes)')



In [ ]:
# Basic QA: confirm all four figure files exist, raster images are non-empty, and the SVG contains no Chinese characters.
# This does not modify any outputs; it only checks rerunnability and file format.
expected_figure_files = [Path(f'{FIG_BASENAME}.{suffix}') for suffix in ['svg', 'pdf', 'tiff', 'png']]
missing_figure_files = [p for p in expected_figure_files if not p.exists() or p.stat().st_size == 0]
if missing_figure_files:
    raise FileNotFoundError(f'Figure export failed: {missing_figure_files}')

for raster_path in [Path(f'{FIG_BASENAME}.png'), Path(f'{FIG_BASENAME}.tiff')]:
    with Image.open(raster_path) as img:
        arr = np.asarray(img.convert('RGB'))
        non_white_ratio = np.mean(np.any(arr < 245, axis=2))
        print(f'{raster_path.name}: size={img.size}, non_white_ratio={non_white_ratio:.4f}')
        if non_white_ratio < 0.01:
            raise ValueError(f'{raster_path.name} may be a blank figure.')

svg_text = Path(f'{FIG_BASENAME}.svg').read_text(encoding='utf-8')
if re.search(r'[一-鿿]', svg_text):
    raise ValueError('Chinese characters detected in SVG; check that all figure text is English.')

print('Figure QA passed.')


In [ ]:
# Save the method notes.
# The log records input data, keyword table, five-dimensional framework, category counts, application-maturity derivation rules, output files, and basic QA results.
category_counts_text = '\n'.join(
    f"- {row.dimension_en} ({row.dimension_cn}): {int(row.n_categories)} categories"
    for row in dimension_summary.itertuples(index=False)
)

maturity_counts_text = '\n'.join(
    f"- {label}: {count} records ({count / n_records * 100:.1f}%)"
    for label, count in term_df['application_maturity'].value_counts().items()
)

method_nonempty = int(term_df['method_term_set'].apply(bool).sum())
object_nonempty = int(term_df['object_term_set'].apply(bool).sum())
performance_nonempty = int(term_df['performance_term_set'].apply(bool).sum())
all_three = int((term_df['method_term_set'].apply(bool) & term_df['object_term_set'].apply(bool) & term_df['performance_term_set'].apply(bool)).sum())

log_text = f"""# Subagent 04 Coding System and Conceptual Framework Method Notes

## Input Data
- Master data: `data/{MAIN_DATA_FILENAME}`
- Keyword table: `data/{KEYWORD_TABLE_FILENAME}` ({len(keyword_df)} keywords)
- Coding denominator: N={n_records}
- Year range: {year_min}-{year_max}
- Fields used: `matched_method_terms`, `matched_object_terms`, `matched_performance_terms`

## Term-Hit Structure
- Non-empty method/data terms: {method_nonempty}/{n_records}
- Non-empty built-environment object terms: {object_nonempty}/{n_records}
- Non-empty environmental performance terms: {performance_nonempty}/{n_records}
- All three term groups non-empty: {all_three}/{n_records}

## Five-Dimensional Coding Framework
This notebook builds a five-dimensional coding framework: Built-environment object, Sensing data source, Analytical method, Environmental performance, and Application maturity. The first four dimensions are non-exclusive keyword hits, with all keywords drawn from the new 37-keyword table; one record can hit multiple categories within the same dimension. The fifth dimension is an exclusive derived primary stage used to describe application maturity.

{category_counts_text}

## Application-Maturity Derivation Rules
Application maturity is not an existing field in the raw data; it is derived from combinations of the three matched-term groups. Priority from high to low is:
1. Decision support and planning application: object terms include “规划” with performance terms present, or “海绵” co-occurs with “洪涝/韧性”.
2. Model-based prediction and mechanism inference: method terms include AI/机器学习/深度学习/三维/LiDAR/多源数据 and object or performance terms are present.
3. Performance measurement and assessment: object and performance terms both appear, and no higher-priority rule above is matched.
4. Built-environment mapping and characterization: object terms are present but performance terms are absent.
5. Exploratory sensing and data inventory: only method/data terms are present, with no explicit object or performance terms.

Application-maturity stage distribution:
{maturity_counts_text}

## Fig. 3 Figure Design
Fig. 3 uses vertically stacked modules to show the five-layer conceptual framework: sensing data sources, analytical methods, built environment objects, environmental performance, and application maturity. The figure uses unfilled modules and chips, black outlines, and enlarged English labels; SVG/PDF use transparent backgrounds, while PNG/TIFF use a white canvas for raster previews.

## Output Files
- Coding-framework table: `output/tables/04_coding_framework.csv`
- Conceptual framework SVG: `output/figures/Fig3_conceptual_framework.svg`
- Conceptual framework PDF: `output/figures/Fig3_conceptual_framework.pdf`
- Conceptual framework TIFF: `output/figures/Fig3_conceptual_framework.tiff`
- Conceptual framework PNG: `output/figures/Fig3_conceptual_framework.png`
- Method notes: `output/logs/04_coding_methods.md`

## Rerunnability and QA
- The notebook can run from scratch from either the project root or the `code/` directory.
- After figure export, SVG/PDF/TIFF/PNG are checked for existence and non-empty size.
- PNG/TIFF pass a non-empty pixel check.
- SVG passes a Chinese-character check to ensure figure text is English.

Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

METHOD_LOG.write_text(log_text, encoding='utf-8')
print(f'Saved method notes: {METHOD_LOG}')
print(log_text[:1000])


In [ ]:
# Final output list for quick checks at the end of a notebook run.
outputs = [
    FRAMEWORK_CSV,
    Path(f'{FIG_BASENAME}.svg'),
    Path(f'{FIG_BASENAME}.pdf'),
    Path(f'{FIG_BASENAME}.tiff'),
    Path(f'{FIG_BASENAME}.png'),
    METHOD_LOG,
]

for p in outputs:
    if not p.exists() or p.stat().st_size == 0:
        raise FileNotFoundError(f'Output file is missing or empty: {p}')
    print(f'{p.relative_to(PROJECT_ROOT)}\t{p.stat().st_size:,} bytes')

print('Subagent 04 notebook completed successfully.')